In [5]:
from sklearn.random_projection import johnson_lindenstrauss_min_dim, SparseRandomProjection
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn import metrics
from sklearn import datasets
from sklearn.datasets import fetch_20newsgroups_vectorized
import matplotlib.pyplot as plt
import numpy as np
import warnings
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import fashion_mnist, cifar10
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score
import time
from joblib import Parallel, delayed
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')


In [22]:
def compute_baseline_accuracy(given_model,train_data, train_target, test_data, test_target):
    """
    Train a classifier on full-dimensional data and compute accuracy.

    This establishes a baseline for comparison with dimensionality reduction methods.
    Uses LinearSVC, a support vector machine for classification.

    Parameters:
    - train_data, train_target: Training features and labels
    - test_data, test_target: Test features and labels

    Returns:
    - accuracy: Classification accuracy score (0 to 1)
    """
    model = given_model()
    model.fit(train_data, train_target)
    predictions = model.predict(test_data)
    return metrics.accuracy_score(predictions, test_target)

def _single_svc_task(train_data, train_target, test_data, test_target, dim):
    """Execution unit for one dimension and one run."""
    # 1. Projection
    projection = SparseRandomProjection(n_components=dim)
    xtr = projection.fit_transform(train_data)
    xte = projection.transform(test_data)

    # 2. LinearSVC Optimization
    # 'dual=False' is preferred when n_samples > n_features (typical for RP)
    # 'tol=1e-3' and 'max_iter=2000' prevent hanging on slow convergence
    model = LinearSVC(dual=False, max_iter=2000, tol=1e-3)
    model.fit(xtr, train_target)

    return model.score(xte, test_target)

def evaluate_svc_projections_final(train_data, train_target, test_data, test_target,
                                   dimensions, n_runs=10, n_jobs=-1):

    # Flattened task list: [dim1_run1, dim1_run2, ... dimN_runM]
    # n_jobs=-1 uses all available CPU logical cores
    flat_results = Parallel(n_jobs=n_jobs)(
        delayed(_single_svc_task)(train_data, train_target, test_data, test_target, d)
        for d in dimensions for _ in range(n_runs)
    )

    # Reshape to (number_of_dimensions, n_runs)
    results_matrix = np.array(flat_results).reshape(len(dimensions), n_runs)

    # Average across the runs (rows) to get one value per dimension
    return results_matrix.mean(axis=1)

def _evaluate_single_instance(train_data, train_target, test_data, test_target, dim):
    """Worker function for one specific dimension and one specific run."""
    # 1. Project
    projection = SparseRandomProjection(n_components=dim)
    projected_train = projection.fit_transform(train_data)
    projected_test = projection.transform(test_data)

    # 2. KNN Logic
    # For KNN, lowering max_samples or using 'kd_tree' speeds up lower dims
    model = KNeighborsClassifier(n_neighbors=5, algorithm='auto')
    model.fit(projected_train, train_target)

    return model.score(projected_test, test_target)

def evaluate_knn_projections_final(train_data, train_target, test_data, test_target,
                                   dimensions, n_runs=10, n_jobs=-1):

    # Create a flat list of tasks: [(dim1, run1), (dim1, run2)... (dimN, runN)]
    # This ensures all CPU cores stay busy until the very last task is done
    all_accuracies = Parallel(n_jobs=n_jobs)(
        delayed(_evaluate_single_instance)(train_data, train_target, test_data, test_target, d)
        for d in dimensions for _ in range(n_runs)
    )

    # Reshape and average
    # Final shape: (len(dimensions), n_runs)
    results_matrix = np.array(all_accuracies).reshape(len(dimensions), n_runs)

    return results_matrix.mean(axis=1)

def plot_accuracy_comparison(dimensions, baseline_accuracy, projected_accuracies,
                           xlim=(2, 2000), dataset_name="Dataset"):
    """
    Visualize classification accuracy vs projection dimension.

    Plots two lines: baseline accuracy (red) for full-dimensional data,
    and accuracy with random projections (green) for various reduced dimensions.

    Parameters:
    - dimensions: Array of k values (projected dimensions)
    - baseline_accuracy: Reference accuracy from full-dimensional classifier
    - projected_accuracies: Accuracies achieved at each dimension
    - xlim: X-axis limits for the plot (default (2, 2000))
    - dataset_name: Name of dataset for plot title
    """
    plt.figure()
    plt.xlabel("Number of dimensions k")
    plt.ylabel("Accuracy")
    plt.xlim(xlim)
    plt.ylim([0, 1])
    plt.title(f"Accuracy Comparison for {dataset_name}")

    # Plot baseline
    plt.plot(dimensions, [baseline_accuracy] * len(projected_accuracies),
             color="r", label="Baseline")

    # Plot projected accuracies
    plt.plot(dimensions, projected_accuracies, color="g", label="Random Projection")

    plt.legend()
    plt.show()

In [8]:
newsgroups = fetch_20newsgroups_vectorized(subset='all')
print("=== 20 Newsgroups (Vectorized) ===")
print("Shape:", newsgroups.data.shape)
print("Number of samples:", newsgroups.data.shape[0])
print("Number of features:", newsgroups.data.shape[1])
train_data, test_data, train_target, test_target = train_test_split(newsgroups.data, newsgroups.target)

=== 20 Newsgroups (Vectorized) ===
Shape: (18846, 130107)
Number of samples: 18846
Number of features: 130107


In [15]:
(X_train_f, y_train_f), (X_test_f, y_test_f) = fashion_mnist.load_data()
X_train_f_flat = X_train_f.reshape(X_train_f.shape[0], -1) / 255.0
X_test_f_flat = X_test_f.reshape(X_test_f.shape[0], -1) / 255.0
print("Shape:", X_train_f_flat.shape)
print("Number of samples:", X_train_f_flat.shape[0])
print("Number of features:", X_train_f_flat.shape[1])

Shape: (60000, 784)
Number of samples: 60000
Number of features: 784


In [12]:
baseline_news = compute_baseline_accuracy(LinearSVC,train_data, train_target,test_data, test_target)
baseline_fmnist = compute_baseline_accuracy(KNeighborsClassifier,X_train_f_flat, y_train_f,X_test_f_flat, y_test_f)

In [13]:
n_samples = newsgroups.data.shape[0]
print(f"Number of samples: {n_samples}")
print(f"Johnson-Lindenstrauss minimum dimensions (eps=0.1): {johnson_lindenstrauss_min_dim(n_samples, eps=0.1)}")

Number of samples: 18846
Johnson-Lindenstrauss minimum dimensions (eps=0.1): 8437


In [19]:
n_samples = X_test_f_flat.shape[0]
print(f"Number of samples: {n_samples}")
print(f"Johnson-Lindenstrauss minimum dimensions (eps=0.1): {johnson_lindenstrauss_min_dim(n_samples, eps=0.4)}")

Number of samples: 10000
Johnson-Lindenstrauss minimum dimensions (eps=0.1): 627


In [ ]:
dims_news = [1000,2000,4000,8000,16000,32000]
dims_fmnist = [25,50,100,200,400,700]

projected_accuracies = evaluate_svc_projections_final(train_data.tocsr(), train_target,test_data, test_target,dims_news,n_runs=10)